# 4-2. PII 마스킹 · 가드레일 · 감사 로그

⏱ **소요시간**: 30분

## 학습 목표

- 🔒 **Presidio**로 개인정보(PII)를 탐지·가명처리한다.
- 한국 금융 실무용 커스텀 인식기(주민등록번호 RRN, 계좌번호 ACCOUNT, 사업자등록번호 BRN)를 구현한다.
- `data/pii_samples.jsonl` 기반 **자동 평가 루프**로 recall을 측정한다.
- OWASP **LLM Top 10** 위협을 이해하고, 🔒 **NeMo Guardrails**로 입출력 필터링 rail을 구성한다.
- LLM 호출 **감사 로그**(JSONL) 유틸을 작성한다 — 전자금융감독규정 §15의 로그 보존 요건 대응.

## 핵심 키워드

`Presidio` · `PatternRecognizer` · `PII` · `RRN` · `BRN` · `가명처리` · `NeMo Guardrails` · `Colang` · `OWASP LLM Top10` · `감사로그` · `트레이스`

## 배지

- 🔒 Presidio (로컬 실행, 외부 호출 없음)
- 🔒 NeMo Guardrails (로컬 실행, LLM은 교체 가능)
- 🔒 감사 로그 (내부 파일시스템 또는 Kafka)

In [ ]:
# 공통 헬퍼 로드
import sys
sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge

print(provider_badge())

## 1. Presidio 기본: analyze / anonymize

Presidio는 Microsoft 오픈소스 PII 탐지·비식별 라이브러리이다.

- `AnalyzerEngine`: 텍스트 → `RecognizerResult[]` (entity_type, start, end, score)
- `AnonymizerEngine`: 탐지 결과를 바탕으로 마스킹/대체/해시
- `RecognizerRegistry`: 커스텀 인식기 등록

```bash
pip install presidio-analyzer presidio-anonymizer
python -m spacy download en_core_web_sm
```

In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

sample = "Contact John Doe at john@example.com or +1-202-555-0173"
results = analyzer.analyze(text=sample, language="en")
for r in results:
    print(r)

masked = anonymizer.anonymize(text=sample, analyzer_results=results)
print("\n[MASKED]", masked.text)

## 2. 한국어 커스텀 인식기

Presidio 기본 인식기는 영어/유럽권 위주이므로, 한국 금융 PII는 **PatternRecognizer**로 직접 만든다.

| 엔티티 | 포맷 예시 | 정규식 근거 |
| --- | --- | --- |
| `KR_RRN` (주민등록번호) | `900101-1234567` | 6자리-7자리, 뒷자리 첫 숫자 1~4 |
| `KR_ACCOUNT` (계좌번호) | `123-456-7890123` | 3~6자리 세그먼트 2~3개, 총 10~16자리 |
| `KR_BRN` (사업자등록번호) | `123-45-67890` | 3-2-5 형태 |
| `KR_PHONE` | `010-1234-5678` | 휴대전화 |
| `KR_CARD` | `4000-1234-5678-9010` | 카드번호 4-4-4-4 |

> 실무에서는 정규식만으로는 위험하다 — **체크섬 검증**(RRN, BRN), **BIN 테이블 확인**(카드), **NER 앙상블**을 추가한다. 본 수업은 교보재 수준.

In [ ]:
def build_korean_recognizers() -> list[PatternRecognizer]:
    """한국 금융 PII 커스텀 인식기 팩토리."""
    rrn = PatternRecognizer(
        supported_entity="KR_RRN",
        name="KR_RRN_Recognizer",
        supported_language="ko",
        patterns=[Pattern(name="rrn_dash", regex=r"\b\d{6}-[1-4]\d{6}\b", score=0.95)],
        context=["주민", "주민등록", "주민번호"],
    )
    account = PatternRecognizer(
        supported_entity="KR_ACCOUNT",
        name="KR_ACCOUNT_Recognizer",
        supported_language="ko",
        patterns=[Pattern(name="acct_dash", regex=r"\b\d{2,6}-\d{2,6}-\d{2,7}\b", score=0.7)],
        context=["계좌", "입금", "송금", "예금주"],
    )
    brn = PatternRecognizer(
        supported_entity="KR_BRN",
        name="KR_BRN_Recognizer",
        supported_language="ko",
        patterns=[Pattern(name="brn_dash", regex=r"\b\d{3}-\d{2}-\d{5}\b", score=0.85)],
        context=["사업자", "사업자등록", "세금계산서"],
    )
    phone = PatternRecognizer(
        supported_entity="KR_PHONE",
        name="KR_PHONE_Recognizer",
        supported_language="ko",
        patterns=[Pattern(name="phone_mobile", regex=r"\b01[016789]-\d{3,4}-\d{4}\b", score=0.9)],
        context=["연락처", "전화", "휴대폰"],
    )
    card = PatternRecognizer(
        supported_entity="KR_CARD",
        name="KR_CARD_Recognizer",
        supported_language="ko",
        patterns=[Pattern(name="card16", regex=r"\b\d{4}-\d{4}-\d{4}-\d{4}\b", score=0.9)],
        context=["카드", "신용카드"],
    )
    return [rrn, account, brn, phone, card]


# 한국어 지원 AnalyzerEngine 구성 (NLP 엔진은 영어 기반이지만 PatternRecognizer는 언어 독립)
from presidio_analyzer.nlp_engine import NlpEngineProvider

nlp_conf = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
    # 한국어 spaCy 모델이 환경에 없을 수 있어 en 모델만 등록, ko는 PatternRecognizer로 커버
}
provider = NlpEngineProvider(nlp_configuration=nlp_conf)
nlp_engine = provider.create_engine()

ko_analyzer = AnalyzerEngine(
    nlp_engine=nlp_engine,
    supported_languages=["en", "ko"],
)
for rec in build_korean_recognizers():
    ko_analyzer.registry.add_recognizer(rec)

print("등록된 인식기 수:", len(ko_analyzer.registry.recognizers))

In [ ]:
# 간단 동작 확인
text = "홍길동 고객님의 주민등록번호는 900101-1234567, 계좌번호 123-456-7890123입니다."
results = ko_analyzer.analyze(text=text, language="ko")
for r in results:
    print(f"{r.entity_type:15s} {text[r.start:r.end]!r:30s} score={r.score:.2f}")

masked = anonymizer.anonymize(
    text=text,
    analyzer_results=results,
    operators={
        "KR_RRN": OperatorConfig("replace", {"new_value": "[RRN]"}),
        "KR_ACCOUNT": OperatorConfig("mask", {"chars_to_mask": 8, "masking_char": "*", "from_end": True}),
    },
)
print("\n[MASKED]", masked.text)

## 3. 자동 평가 루프: pii_samples.jsonl

`data/pii_samples.jsonl`의 각 샘플에는 기대되는 마스킹 엔티티 레이블(`expected_masks`)이 들어 있다.
우리 인식기가 탐지한 엔티티를 **내부 라벨 체계**에 매핑한 후 recall을 계산한다.

In [ ]:
import json
from pathlib import Path

# 엔티티 레이블 매핑: Presidio → 수업용 레이블
ENTITY_MAP = {
    "KR_RRN": "RRN",
    "KR_ACCOUNT": "ACCOUNT",
    "KR_BRN": "BRN",
    "KR_PHONE": "PHONE",
    "KR_CARD": "CREDIT_CARD",
    "PHONE_NUMBER": "PHONE",
    "CREDIT_CARD": "CREDIT_CARD",
    "EMAIL_ADDRESS": "EMAIL",
    "PERSON": "PERSON",
    "LOCATION": "ADDRESS",
}

SAMPLES_PATH = Path("../../data/pii_samples.jsonl")
samples = [json.loads(line) for line in SAMPLES_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"로드한 샘플 수: {len(samples)}")
samples[:2]

In [ ]:
def detect_labels(text: str) -> set[str]:
    """텍스트에서 탐지된 내부 라벨 집합을 반환."""
    results = ko_analyzer.analyze(text=text, language="ko")
    labels = set()
    for r in results:
        mapped = ENTITY_MAP.get(r.entity_type)
        if mapped:
            labels.add(mapped)
    return labels


total_expected = 0
total_hit = 0
print(f"{'id':>3} | {'expected':30s} | {'detected':30s} | recall")
print("-" * 90)
for s in samples:
    expected = set(s["expected_masks"])
    detected = detect_labels(s["text"])
    hit = len(expected & detected)
    total_expected += len(expected)
    total_hit += hit
    recall = f"{hit}/{len(expected)}" if expected else "n/a"
    print(f"{s['id']:>3} | {str(sorted(expected))[:28]:30s} | {str(sorted(detected))[:28]:30s} | {recall}")

overall = total_hit / total_expected if total_expected else 0
print(f"\n[전체 recall] {total_hit}/{total_expected} = {overall:.1%}")

> 관찰: 정규식 기반 인식기는 RRN/BRN/CREDIT_CARD 등 **정형 식별자**에는 강하지만 PERSON/ADDRESS/EMP_ID/DRIVER_LICENSE 같은 **문맥 의존** 엔티티는 놓칠 수 있다. 실무에서는 한국어 NER(`klue/roberta`, `KoELECTRA-NER`) 또는 LLM 기반 보조 분류기를 추가한다.

## 4. OWASP Top 10 for LLM Applications (요약)

| # | 위협 | 한 줄 요약 | 본 과정 대응 |
| --- | --- | --- | --- |
| LLM01 | Prompt Injection | 외부 텍스트가 시스템 지시를 덮어씀 | NeMo 입력 rail, 문서 메타데이터 필터 |
| LLM02 | Insecure Output Handling | 모델 출력이 downstream에서 RCE/SSRF | 출력 sanitizer, JSON 스키마 검증 |
| LLM03 | Training Data Poisoning | 학습 데이터 오염 | 데이터 출처 관리, SBOM |
| LLM04 | Model Denial of Service | 토큰/컨텍스트 폭탄 | rate limit, max_tokens |
| LLM05 | Supply Chain Vulnerabilities | 악성 모델/라이브러리 | 모델 sha256 검증, 오프라인 번들 |
| LLM06 | Sensitive Information Disclosure | PII·기밀 유출 | Presidio, 출력 rail, 감사로그 |
| LLM07 | Insecure Plugin Design | 툴/함수 호출 권한 과다 | Allow-list, scope별 토큰 |
| LLM08 | Excessive Agency | 에이전트가 과도한 자동 실행 | Human-in-the-loop 승인 |
| LLM09 | Overreliance | 환각을 그대로 신뢰 | 출처 인용, 신뢰도 점수 |
| LLM10 | Model Theft | 모델 가중치 유출 | 접근통제, 망분리 |

## 5. NeMo Guardrails로 입출력 필터링 rail 구성

```bash
pip install nemoguardrails
```

NeMo Guardrails는 **Colang** DSL과 설정 파일로 대화 흐름 가드레일을 정의한다.

- **Input rail**: 사용자 입력을 분류해 금지어/PII/탈옥 시도 차단
- **Output rail**: 모델 출력을 후처리해 PII 재유출, 금지 주제 차단
- **Dialog rail**: flow 단위로 주제 유도

In [ ]:
# 런타임에 config를 메모리로 구성하는 최소 예제
YAML_CONFIG = """
models:
  - type: main
    engine: openai
    model: gpt-4o-mini
rails:
  input:
    flows:
      - block pii input
  output:
    flows:
      - block pii output
"""

COLANG = """
define user ask about rrn
  "주민등록번호"
  "주민번호"

define bot refuse pii
  "죄송합니다. 개인정보(주민등록번호 등)를 포함한 요청은 처리할 수 없습니다."

define flow block pii input
  user ask about rrn
  bot refuse pii
  stop

define flow block pii output
  bot ...
  $contains_pii = execute check_output_pii(text=$last_bot_message)
  if $contains_pii
    bot refuse pii
    stop
"""

print(YAML_CONFIG)
print(COLANG)

In [ ]:
# 실행 예시 (환경에 nemoguardrails 가 설치되어 있어야 함)
try:
    from nemoguardrails import LLMRails, RailsConfig

    config = RailsConfig.from_content(yaml_content=YAML_CONFIG, colang_content=COLANG)
    rails = LLMRails(config)

    # 커스텀 액션: 출력 PII 체크 (Presidio 재사용)
    async def check_output_pii(text: str) -> bool:
        return bool(ko_analyzer.analyze(text=text, language="ko"))

    rails.register_action(check_output_pii, name="check_output_pii")

    resp = rails.generate(messages=[{"role": "user", "content": "내 주민등록번호는 900101-1234567이야. 확인해줘"}])
    print("[rail 응답]", resp)
except Exception as e:
    print("NeMo Guardrails 설치/실행 생략:", type(e).__name__, str(e)[:200])
    print("\n→ 실습 환경에서는 `pip install nemoguardrails` 후 재실행하세요.")

## 6. 감사 로그 JSONL 유틸

전자금융감독규정 §15(로그 보관 5년)와 금감원 AI 활용 안내서(추적가능성)에 대응하기 위해,
모든 LLM 호출은 다음 스키마로 append-only JSONL에 기록한다.

```json
{
  "timestamp": "2025-04-17T10:23:45+09:00",
  "trace_id": "uuid4",
  "user": "emp_0042",
  "role": "analyst",
  "query": "...",
  "context_sources": [{"doc_id": "FAQ-007", "score": 0.81}],
  "response": "...",
  "pii_flags": ["KR_RRN"],
  "model": "qwen2.5:7b-instruct",
  "provider": "ollama",
  "latency_ms": 1234
}
```

> 운영에서는 JSONL → Kafka → Elastic/Opensearch 또는 SIEM(Splunk 등)으로 흘려보낸다.

In [ ]:
import json, os, time, uuid
from datetime import datetime, timezone, timedelta
from pathlib import Path

KST = timezone(timedelta(hours=9))
AUDIT_PATH = Path("./audit.jsonl")


def audit_log(
    user: str,
    role: str,
    query: str,
    response: str,
    context_sources: list[dict] | None = None,
    model: str = "unknown",
    provider: str = "unknown",
    latency_ms: int = 0,
    path: Path = AUDIT_PATH,
) -> dict:
    """LLM 호출 감사 로그를 append-only JSONL로 기록."""
    pii_flags = sorted({r.entity_type for r in ko_analyzer.analyze(text=f"{query}\n{response}", language="ko")})
    record = {
        "timestamp": datetime.now(KST).isoformat(timespec="seconds"),
        "trace_id": str(uuid.uuid4()),
        "user": user,
        "role": role,
        "query": query,
        "context_sources": context_sources or [],
        "response": response,
        "pii_flags": pii_flags,
        "model": model,
        "provider": provider,
        "latency_ms": latency_ms,
    }
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record


# 데모 호출
t0 = time.time()
demo_query = "고객 홍길동(900101-1234567)님 계좌 조회 방법?"
demo_response = "보안상 주민등록번호를 직접 받지 않습니다. 본인 인증 절차로 안내드립니다."
rec = audit_log(
    user="emp_0042",
    role="cs_agent",
    query=demo_query,
    response=demo_response,
    context_sources=[{"doc_id": "FAQ-001", "score": 0.77}],
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    provider=os.getenv("LLM_PROVIDER", "openai"),
    latency_ms=int((time.time() - t0) * 1000),
)
print(json.dumps(rec, ensure_ascii=False, indent=2))

In [ ]:
# 기록된 로그 읽어 확인
for line in AUDIT_PATH.read_text(encoding="utf-8").splitlines()[-3:]:
    print(line)